# Exercise 2: Proof of Work - Mining, Difficulty & Probability

In [ ]:
import hashlib
import time
import random
import statistics

## Proof of Work Mining Function

In [ ]:
def mine_block(block_data: str, difficulty: int):
    """
    Mines a block by incrementing a nonce until the SHA-256 hash
    of (block_data + nonce) begins with `difficulty` leading zeros.

    Returns: (valid_hash, nonce, elapsed_time, hash_attempts)
    """
    target_prefix = "0" * difficulty
    nonce = 0
    attempts = 0
    start_time = time.perf_counter()

    while True:
        attempts += 1
        input_str = f"{block_data}{nonce}"
        hash_result = hashlib.sha256(input_str.encode()).hexdigest()

        if hash_result.startswith(target_prefix):
            elapsed_time = time.perf_counter() - start_time
            return hash_result, nonce, elapsed_time, attempts

        nonce += 1

## Experiment: Mine 30 blocks at each difficulty level

In [ ]:
def run_experiment(difficulties, blocks_per_difficulty=30):
    results = {}

    for difficulty in difficulties:
        print(f"\n=== Mining {blocks_per_difficulty} blocks at difficulty {difficulty} ===")
        times = []
        attempts_list = []

        for i in range(blocks_per_difficulty):
            block_data = f"Block data #{i} - random:{random.random()}"
            valid_hash, nonce, elapsed_time, attempts = mine_block(block_data, difficulty)

            times.append(elapsed_time)
            attempts_list.append(attempts)

            print(f"Block {i+1:2d}: hash={valid_hash[:16]}..., "
                  f"nonce={nonce}, time={elapsed_time:.6f}s, attempts={attempts}")

        results[difficulty] = {
            "times": times,
            "attempts": attempts_list,
        }

    return results

## Performance Metrics Calculation

In [ ]:
def summarize_results(results):
    print("\n\n=== Performance Summary ===")
    header = (f"{'Difficulty':<10} {'AvgTime(s)':<12} {'MinTime(s)':<12} "
              f"{'MaxTime(s)':<12} {'StdDev(s)':<12} {'Throughput(H/s)':<16} "
              f"{'AvgAttempts':<14} {'TheoreticalAttempts':<20}")
    print(header)
    print("-" * len(header))

    summary_table = []

    for difficulty, data in results.items():
        times = data["times"]
        attempts = data["attempts"]

        avg_time = statistics.mean(times)
        min_time = min(times)
        max_time = max(times)
        std_time = statistics.stdev(times) if len(times) > 1 else 0.0

        avg_attempts = statistics.mean(attempts)
        throughput = avg_attempts / avg_time if avg_time > 0 else 0.0

        theoretical_attempts = 16 ** difficulty

        row = (difficulty, avg_time, min_time, max_time, std_time,
               throughput, avg_attempts, theoretical_attempts)
        summary_table.append(row)

        print(f"{difficulty:<10} {avg_time:<12.6f} {min_time:<12.6f} "
              f"{max_time:<12.6f} {std_time:<12.6f} {throughput:<16.2f} "
              f"{avg_attempts:<14.2f} {theoretical_attempts:<20}")

    return summary_table

## Run the Experiment

In [ ]:
difficulties_to_test = [2, 3, 4, 5]

results = run_experiment(difficulties_to_test, blocks_per_difficulty=30)

## Summary Table: Measured vs Theoretical

In [ ]:
summary_table = summarize_results(results)